# Parity-Net Colab Runner
This notebook clones the GitHub repo, trains the mean-field-style residual network from `MOTIVATION.md`, saves checkpoints to Google Drive, and runs PCA rank-reduction analysis.
Before running, set `GITHUB_REPO_URL` to the URL of your pushed repo.



In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
GITHUB_REPO_URL = "https://github.com/labofdoubt/feature-learning-parity-task.git"
REPO_DIR = "/content/feature-learning-parity-task"

!rm -rf "$REPO_DIR"
!git clone "$GITHUB_REPO_URL"
%cd "$REPO_DIR"
!pip install -e .


Cloning into 'feature-learning-parity-task'...
remote: Enumerating objects: 65, done.
remote: Counting objects: 100% (65/65), done.
remote: Compressing objects: 100% (41/41), done.
remote: Total 65 (delta 31), reused 58 (delta 24), pack-reused 0 (from 0)
Receiving objects: 100% (65/65), 398.79 KiB | 1.77 MiB/s, done.
Resolving deltas: 100% (31/31), done.
/content/feature-learning-parity-task
Obtaining file:///content/feature-learning-parity-task
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for parity-net (pyproject.toml) ... done
  Created wheel for parity-net: filename=parity_net-0.1.0-0.editable-py3-none-any.whl size=3070 sha256=7d332c2270a13659d8f3839425adbed8797df0962d7ddb9ee1494a163a3c827c
  Stored in directory: /tmp/pip-ephem-wheel-cache-ehqlg11t/wheels/a2/05/6b/5da726f55833c1e289bd27eaf6964cb7ead0a8

In [4]:
DRIVE_RUN_DIR = "/content/drive/MyDrive/ml_projects_new/parity_runs/example_run_1"

## Create Configs
`barrier_c` is in the training config because it enters as a loss regularizer. If left as `None`, the training code uses `7 / N`.



In [12]:
from pathlib import Path
import yaml

N = 128

config = {
    "model": {
        "input_dim": 32,
        "relevant_dim": 16,
        "N": N,
        "L": 1,
        "activation": "relu",
        "use_readout_barrier": False,
        "embedding_weight_variance": 1.0/32,
        "hidden_weight_variance": 1.0/N,
        "readout_weight_variance": 1.0/N,
        "use_post_activation_linear": False,
        "bias": False,
    },
    "training": {
        "num_steps": 100_000,
        "test_samples": 100_000,
        "batch_size": 256,
        "seed": 0,
        "device": "cuda",
        "dtype": "float32",
        "log_every": 1_000,
        "checkpoint_every": 1_000,
        "output_dir": DRIVE_RUN_DIR,
        "barrier_c": None,
        "barrier_lambda": 10.0,
        "optimizer": {
            "name": "sgd",
            "lr": 1e-2,
            "weight_decay": 0.0,
            "momentum": 0.9,
            "betas": [0.9, 0.999],
        },
    },
}
config_path = Path(DRIVE_RUN_DIR) / "config.yaml"
config_path.parent.mkdir(parents=True, exist_ok=True)
with config_path.open("w") as f:
    yaml.safe_dump(config, f, sort_keys=False)
config_path




PosixPath('/content/drive/MyDrive/ml_projects_new/parity_runs/example_run_1/config.yaml')

## Train

Training samples a fresh random batch at every optimizer step. `num_steps` is the number of weight updates.
This runs the repo command and saves intermediate checkpoints to Google Drive.



In [ ]:
!parity-train --config "$config_path"

{'step': 1000, 'elapsed_seconds': 2.581937395999944, 'train_mse': 0.4542004466056824, 'barrier': 0.0, 'loss': 0.4542004466056824, 'test_mse': 0.45594966411590576, 'test_mse_d2': 0.021067049354314804, 'test_mse_d4': 0.9097380042076111, 'test_mse_d8': 1.0104496479034424, 'test_mse_d16': 1.0108567476272583}
{'step': 2000, 'elapsed_seconds': 5.15596261799999, 'train_mse': 0.2646001875400543, 'barrier': 0.0, 'loss': 0.2646001875400543, 'test_mse': 0.26652151346206665, 'test_mse_d2': 0.01339640561491251, 'test_mse_d4': 0.2175125926733017, 'test_mse_d8': 1.0070050954818726, 'test_mse_d16': 1.0065910816192627}
{'step': 3000, 'elapsed_seconds': 7.797428898000021, 'train_mse': 0.20732063055038452, 'barrier': 0.0, 'loss': 0.20732063055038452, 'test_mse': 0.20792517066001892, 'test_mse_d2': 0.005821073893457651, 'test_mse_d4': 0.012964450754225254, 'test_mse_d8': 1.0067530870437622, 'test_mse_d16': 1.0069448947906494}
{'step': 4000, 'elapsed_seconds': 10.779463513999985, 'train_mse': 0.20403395593

## Load Checkpoint and Inspect Weight Variance



In [ ]:
import torch
import pandas as pd
from parity_net.checkpoint import load_checkpoint
from parity_net.data import ParityDataset, load_dataset
from parity_net.train import resolve_device, resolve_dtype

checkpoint_path = Path(DRIVE_RUN_DIR) / "checkpoints" / "step_00020000.pt"
device = resolve_device(config["training"]["device"])
dtype = resolve_dtype(config["training"]["dtype"])
model, payload, _ = load_checkpoint(checkpoint_path, device)

saved_output_dir = Path(payload["config"]["training"]["output_dir"])
test_data_path = saved_output_dir / "test_data.pt"
if not test_data_path.exists():
    test_data_path = checkpoint_path.parent.parent / "test_data.pt"
if not test_data_path.exists():
    raise FileNotFoundError(
        f"Could not find saved test data at {test_data_path}. "
        "Rerun training with the updated repo so test_data.pt is saved."
    )

test_data = load_dataset(test_data_path, device, dtype)
print(f"Loaded {test_data.x.shape[0]} saved test samples from {test_data_path}")

weight_variances = model.weight_variances()
pd.DataFrame([
    {"layer": layer, "variance": variance}
    for layer, variance in weight_variances.items()
])


## Inspect Predictions on Test Samples

Run inference on the saved training test samples loaded from `test_data.pt` and compare model outputs with the true parity targets for a few samples.


In [ ]:
num_inspect_samples = 5
inspect_batch_size = 32

inspect_data = ParityDataset(
    x=test_data.x[:inspect_batch_size],
    y=test_data.y[:inspect_batch_size],
)

model.eval()
with torch.no_grad():
    inspect_pred = model(inspect_data.x)

input_columns = [f"x{i}" for i in range(config["model"]["input_dim"])]
output_columns = [
    *[f"d2_{i}" for i in range(8)],
    *[f"d4_{i}" for i in range(4)],
    *[f"d8_{i}" for i in range(2)],
    "d16_0",
]

for sample_idx in range(num_inspect_samples):
    input_sequence = pd.DataFrame(
        [inspect_data.x[sample_idx].detach().cpu().numpy().astype(int)],
        columns=input_columns,
    )
    comparison = pd.DataFrame(
        {
            "target": inspect_data.y[sample_idx].detach().cpu().numpy(),
            "prediction": inspect_pred[sample_idx].detach().cpu().numpy(),
        },
        index=output_columns,
    )
    print(f"Test sample {sample_idx}: input sequence")
    display(input_sequence)
    print(f"Test sample {sample_idx}: targets vs predictions")
    display(comparison)


## Inspect Layerwise Activation Variance

Run inference on a batch from the saved test set and measure, for each sample, the activation variance across the hidden dimension after the embedding and after each residual block.


In [ ]:
activation_batch_size = 256
activation_data = ParityDataset(
    x=test_data.x[:activation_batch_size],
    y=test_data.y[:activation_batch_size],
)

model.eval()
with torch.no_grad():
    _, activations = model(activation_data.x, return_activations=True)

rows = []
for layer_idx, h in enumerate(activations):
    layer_name = "embedding" if layer_idx == 0 else f"block_{layer_idx}"
    sample_variances = h.detach().float().var(dim=1, unbiased=False)
    rows.append(
        {
            "layer_idx": layer_idx,
            "layer": layer_name,
            "mean_sample_variance": sample_variances.mean().item(),
            "std_sample_variance": sample_variances.std(unbiased=False).item(),
            "min_sample_variance": sample_variances.min().item(),
            "max_sample_variance": sample_variances.max().item(),
        }
    )

activation_variance_df = pd.DataFrame(rows)
display(activation_variance_df)


## Neuron-Target Correlations

Compute Pearson correlations between each neuron's activation and each of the 15 tree-parity targets on a saved-test-set subset.


In [ ]:
CORRELATION_SAMPLES = 20_000

correlation_count = min(CORRELATION_SAMPLES, test_data.x.shape[0])
correlation_data = ParityDataset(
    x=test_data.x[:correlation_count],
    y=test_data.y[:correlation_count],
)

model.eval()
with torch.no_grad():
    _, correlation_activations = model(correlation_data.x, return_activations=True)

target_columns = [
    *[f"d2_{i}" for i in range(8)],
    *[f"d4_{i}" for i in range(4)],
    *[f"d8_{i}" for i in range(2)],
    "d16_0",
]

if correlation_data.y.shape[1] != len(target_columns):
    raise ValueError(
        f"Expected {len(target_columns)} target labels, got {correlation_data.y.shape[1]} targets"
    )

y = correlation_data.y.detach().float()
y_centered = y - y.mean(dim=0, keepdim=True)
y_norm = torch.linalg.vector_norm(y_centered, dim=0).clamp_min(torch.finfo(y_centered.dtype).eps)

correlation_by_layer = {}
correlation_rows = []
for layer_idx, h in enumerate(correlation_activations):
    h = h.detach().float()
    h_centered = h - h.mean(dim=0, keepdim=True)
    h_norm = torch.linalg.vector_norm(h_centered, dim=0).clamp_min(torch.finfo(h_centered.dtype).eps)
    corr = (h_centered.T @ y_centered) / (h_norm[:, None] * y_norm[None, :])
    corr_cpu = corr.cpu()
    correlation_by_layer[layer_idx] = corr_cpu
    layer_name = "embedding" if layer_idx == 0 else f"block_{layer_idx}"
    for neuron_idx in range(corr_cpu.shape[0]):
        row = {
            "layer_idx": layer_idx,
            "layer": layer_name,
            "neuron_idx": neuron_idx,
            "max_abs_correlation": corr_cpu[neuron_idx].abs().max().item(),
        }
        row.update({target: corr_cpu[neuron_idx, target_idx].item() for target_idx, target in enumerate(target_columns)})
        correlation_rows.append(row)

neuron_target_correlations = pd.DataFrame(correlation_rows)
print(
    f"Computed correlations for {len(correlation_by_layer)} layers, "
    f"{correlation_by_layer[0].shape[0]} neurons per layer, "
    f"and {len(target_columns)} targets using {correlation_count} saved test samples."
)
display(neuron_target_correlations.head())

# A quick index of the strongest single neuron-target correlations.
display(
    neuron_target_correlations
    .sort_values("max_abs_correlation", ascending=False)
    .head(20)
)


In [ ]:
SELECT_LAYER_IDX = 0
SELECT_NEURON_IDX = 0

if SELECT_LAYER_IDX not in correlation_by_layer:
    raise ValueError(f"SELECT_LAYER_IDX must be one of {list(correlation_by_layer)}")
selected_corr = correlation_by_layer[SELECT_LAYER_IDX]
if SELECT_NEURON_IDX < 0 or SELECT_NEURON_IDX >= selected_corr.shape[0]:
    raise ValueError(f"SELECT_NEURON_IDX must be in [0, {selected_corr.shape[0] - 1}]")

selected_neuron_correlations = pd.DataFrame(
    {
        "target": target_columns,
        "correlation": selected_corr[SELECT_NEURON_IDX].numpy(),
    }
)
selected_neuron_correlations["abs_correlation"] = selected_neuron_correlations["correlation"].abs()
selected_neuron_correlations = selected_neuron_correlations.sort_values(
    "abs_correlation",
    ascending=False,
)

layer_name = "embedding" if SELECT_LAYER_IDX == 0 else f"block_{SELECT_LAYER_IDX}"
print(f"Correlations for {layer_name}, neuron {SELECT_NEURON_IDX}")
display(selected_neuron_correlations)

ax = selected_neuron_correlations.sort_values("target").plot(
    x="target",
    y="correlation",
    kind="bar",
    legend=False,
    figsize=(9, 4),
)
ax.axhline(0.0, color="black", linewidth=1)
ax.set_xlabel("Target")
ax.set_ylabel("Pearson correlation")
ax.set_title(f"{layer_name} neuron {SELECT_NEURON_IDX}: target correlations")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


## PCA Rank-Reduction Analysis
Set `INTERVENTION_LAYER_IDX`, `KEEP_PCS_MIN`, `KEEP_PCS_MAX`, and `KEEP_PCS_STEP` below. Layer index `0` is the embedded residual stream before the first residual block; later indices are after residual blocks / before the corresponding next block.


In [ ]:
import matplotlib.pyplot as plt
from parity_net.analysis import (
    collect_layer_activations,
    make_pca_intervention,
    pca_from_activations,
    per_degree_mse,
    predict_in_batches,
    rank_for_threshold,
)

INTERVENTION_LAYER_IDX = 2
KEEP_PCS_MIN = 0
KEEP_PCS_MAX = 70
KEEP_PCS_STEP = 1
PCA_SAMPLES = config["training"]["test_samples"]
ANALYSIS_DIR = Path(DRIVE_RUN_DIR) / "analysis"
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)

analysis_config = payload["config"]
analysis_training = analysis_config["training"]
analysis_model_config = analysis_config["model"]
batch_size = analysis_training["batch_size"]

if PCA_SAMPLES is not None and PCA_SAMPLES < test_data.x.shape[0]:
    heldout = ParityDataset(
        x=test_data.x[:PCA_SAMPLES],
        y=test_data.y[:PCA_SAMPLES],
    )
elif PCA_SAMPLES is not None and PCA_SAMPLES > test_data.x.shape[0]:
    raise ValueError(
        f"PCA_SAMPLES={PCA_SAMPLES} exceeds saved test set size={test_data.x.shape[0]}"
    )
else:
    heldout = test_data
print(f"Using {heldout.x.shape[0]} saved test samples for PCA analysis")

model.eval()
with torch.no_grad():
    baseline_pred = predict_in_batches(model, heldout.x, batch_size)
baseline_metrics = per_degree_mse(baseline_pred, heldout.y)
pd.DataFrame([baseline_metrics]).to_csv(ANALYSIS_DIR / "baseline_mse.csv", index=False)
print("Baseline MSE")
display(pd.DataFrame([baseline_metrics]))

activations = collect_layer_activations(model, heldout.x, batch_size)
pcas = [pca_from_activations(layer_acts) for layer_acts in activations]

rank_rows = []
for layer_idx, pca in enumerate(pcas):
    cumulative = pca["cumulative_explained_variance"]
    rank_rows.append(
        {
            "layer_idx": layer_idx,
            "rank_90": rank_for_threshold(cumulative, 0.90),
            "rank_99": rank_for_threshold(cumulative, 0.99),
            "num_dimensions": cumulative.numel(),
        }
    )
rank_df = pd.DataFrame(rank_rows)
rank_df.to_csv(ANALYSIS_DIR / "pca_rank_thresholds.csv", index=False)
print("PCA rank thresholds")
display(rank_df)

if INTERVENTION_LAYER_IDX < 0 or INTERVENTION_LAYER_IDX >= len(pcas):
    raise ValueError(f"INTERVENTION_LAYER_IDX must be in [0, {len(pcas) - 1}]")
if KEEP_PCS_STEP <= 0:
    raise ValueError("KEEP_PCS_STEP must be positive")
if KEEP_PCS_MIN < 0:
    raise ValueError("KEEP_PCS_MIN must be non-negative")
if KEEP_PCS_MAX < KEEP_PCS_MIN:
    raise ValueError("KEEP_PCS_MAX must be at least KEEP_PCS_MIN")

max_available_pcs = pcas[INTERVENTION_LAYER_IDX]["components"].shape[0]
keep_pcs_max_effective = min(KEEP_PCS_MAX, max_available_pcs)
if keep_pcs_max_effective < KEEP_PCS_MAX:
    print(f"Capping KEEP_PCS_MAX at {keep_pcs_max_effective}; only that many PCs are available.")
if KEEP_PCS_MIN > keep_pcs_max_effective:
    raise ValueError(
        f"KEEP_PCS_MIN={KEEP_PCS_MIN} exceeds available PCs={keep_pcs_max_effective} "
        f"at layer {INTERVENTION_LAYER_IDX}"
    )

keep_pcs_values = list(range(KEEP_PCS_MIN, keep_pcs_max_effective + 1, KEEP_PCS_STEP))
if keep_pcs_values[-1] != keep_pcs_max_effective:
    keep_pcs_values.append(keep_pcs_max_effective)

sweep_rows = []
for keep_pcs in keep_pcs_values:
    intervention = make_pca_intervention(pcas[INTERVENTION_LAYER_IDX], keep_pcs)
    with torch.no_grad():
        pred_intervened = predict_in_batches(
            model,
            heldout.x,
            batch_size,
            intervention=(INTERVENTION_LAYER_IDX, intervention),
        )
    sweep_rows.append(
        {
            "intervention_layer": INTERVENTION_LAYER_IDX,
            "keep_pcs": keep_pcs,
            **per_degree_mse(pred_intervened, heldout.y),
        }
    )

pca_sweep_df = pd.DataFrame(sweep_rows)
pca_sweep_df.to_csv(ANALYSIS_DIR / "pca_intervention_mse_sweep.csv", index=False)
print("PCA intervention MSE sweep")
display(pca_sweep_df)

mse_columns = ["mse_all", "mse_d2", "mse_d4", "mse_d8", "mse_d16"]
ax = pca_sweep_df.plot(
    x="keep_pcs",
    y=mse_columns,
    marker="o",
    figsize=(8, 5),
)
ax.set_xlabel("Number of PCs kept")
ax.set_ylabel("MSE")
ax.set_title(f"PCA intervention at layer {INTERVENTION_LAYER_IDX}")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(ANALYSIS_DIR / "pca_intervention_mse_sweep.png", dpi=150)
plt.show()


## Results
The first table reports how many dimensions recover 90% and 99% of variance at each layer. The second reports MSE after the PCA intervention, including degree 2, 4, 8, and 16 parity groups.



In [12]:
rank_df = pd.read_csv(Path(ANALYSIS_DIR) / "pca_rank_thresholds.csv")
intervention_df = pd.read_csv(Path(ANALYSIS_DIR) / "pca_intervention_mse.csv")
baseline_df = pd.read_csv(Path(ANALYSIS_DIR) / "baseline_mse.csv")
print("PCA rank thresholds")
display(rank_df)
print("Baseline MSE")
display(baseline_df)
print("PCA intervention MSE")
display(intervention_df[["intervention_layer", "keep_pcs", "mse_d2", "mse_d4", "mse_d8", "mse_d16", "mse_all"]])



PCA rank thresholds


,layer_idx,rank_90,rank_99,num_dimensions
0,0,29,32,512
1,1,31,54,512
2,2,36,59,512
3,3,39,63,512
4,4,40,64,512


Baseline MSE


,mse_all,mse_d2,mse_d4,mse_d8,mse_d16
0,0.000562,0.000422,0.000345,0.001037,0.001596


PCA intervention MSE


,intervention_layer,keep_pcs,mse_d2,mse_d4,mse_d8,mse_d16,mse_all
0,2,50,0.022841,0.012645,0.061798,0.048411,0.027021
